# 第 10 章　算法与抽样方法
## Monte Carlo、Bootstrap、MCMC 与交叉验证

::: {.callout-note}
## 本章要点

1. **Monte Carlo 模拟**：路径模拟（GBM）、组合 VaR、期权定价验证
2. **Bootstrap**：置信区间构造、夏普比率 CI、Block Bootstrap 处理时序相关
3. **MCMC 简介**：Metropolis-Hastings 直觉、`pymc` 贝叶斯案例
4. **交叉验证**：k-fold 的陷阱、`TimeSeriesSplit`、前视偏差演示

**定位**：本章是第 11–12 章机器学习模块的**方法论铺垫**。
正确的时序交叉验证是所有金融 ML 回测的基础，
Bootstrap 是没有解析解时的通用推断工具。
:::


## 环境准备


In [ ]:
# ── 第 10 章　算法与抽样方法　─────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

OUTPUT = 'output'
os.makedirs(OUTPUT, exist_ok=True)
RNG = np.random.default_rng(42)
print('环境就绪 ✓')


---

## 1　Monte Carlo 模拟

### 1.1　几何布朗运动（GBM）路径模拟

股票价格的标准模型是几何布朗运动：

$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

离散化：

$$S_{t+1} = S_t \exp\left[(\mu - \frac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t} \cdot Z_t\right], \quad Z_t \sim N(0,1)$$


In [ ]:
# ── 1.1  GBM 路径模拟 ──────────────────────────────────────────────────
S0    = 100.0    # 初始价格
mu    = 0.08     # 年化漂移（预期收益）
sigma = 0.20     # 年化波动率
T     = 1.0      # 模拟期限（年）
dt    = 1/252    # 日度步长
N_steps = int(T / dt)
N_paths = 10_000

# 生成随机增量
Z = RNG.standard_normal((N_paths, N_steps))
log_ret = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z

# 路径：累积乘积
paths = S0 * np.exp(np.cumsum(log_ret, axis=1))
paths = np.hstack([np.full((N_paths, 1), S0), paths])  # 加初始值

print(f'路径数：{N_paths:,}  步数：{N_steps}')
print(f'期末价格统计：均值={paths[:,-1].mean():.2f}  中位数={np.median(paths[:,-1]):.2f}')
print(f'理论期望：S0 * exp(mu*T) = {S0 * np.exp(mu*T):.2f}')

# 可视化（抽取 50 条）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
t = np.linspace(0, T, N_steps+1)
for i in RNG.choice(N_paths, 50, replace=False):
    ax1.plot(t, paths[i], lw=0.4, alpha=0.4, color='steelblue')
ax1.plot(t, paths[:50].mean(axis=0), 'r-', lw=2, label='均值路径')
ax1.set_title(f'GBM 路径模拟（{N_paths:,} 条，显示 50 条）', fontsize=10)
ax1.set_xlabel('时间（年）'); ax1.set_ylabel('价格'); ax1.legend(fontsize=9)

ax2.hist(paths[:,-1], bins=60, color='steelblue', edgecolor='white', density=True)
ax2.axvline(paths[:,-1].mean(), color='red', lw=1.5, label=f'均值={paths[:,-1].mean():.1f}')
ax2.set_title('期末价格分布（对数正态）', fontsize=10)
ax2.set_xlabel('期末价格'); ax2.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch10_gbm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 1.2  组合 VaR：MC 模拟 ────────────────────────────────────────────
# 场景：等权组合（3 只股票），计算 10 天 95% VaR

CONF  = 0.95
DAYS  = 10
N_MC  = 50_000

# 相关系数矩阵（模拟）
corr = np.array([[1.0, 0.6, 0.3],
                  [0.6, 1.0, 0.5],
                  [0.3, 0.5, 1.0]])
vols = np.array([0.20, 0.25, 0.18])     # 年化波动率
daily_vols = vols / np.sqrt(252)

# Cholesky 分解生成相关正态随机数
cov = np.diag(daily_vols) @ corr @ np.diag(daily_vols)
L   = np.linalg.cholesky(cov)

# 模拟 10 天累积收益
Z10   = RNG.standard_normal((N_MC, 3, DAYS))
daily = (L @ Z10).transpose(0,2,1)   # shape: (N_MC, DAYS, 3)
port_daily = daily.mean(axis=2)       # 等权组合
port_10d   = port_daily.sum(axis=1)   # 10 天累积

var_95 = -np.percentile(port_10d, 1 - CONF) * 100
cvar_95= -port_10d[port_10d <= np.percentile(port_10d, 1-CONF)].mean() * 100

print(f'等权组合 {DAYS} 天 VaR（{CONF*100:.0f}%）：{var_95:.2f}%')
print(f'等权组合 {DAYS} 天 CVaR（{CONF*100:.0f}%）：{cvar_95:.2f}%')

# 参数法（方差-协方差法）对比
w = np.ones(3)/3
port_var_daily = w @ cov @ w
var_parametric = stats.norm.ppf(1-CONF) * np.sqrt(port_var_daily * DAYS) * (-100)
print(f'参数法 VaR                     ：{var_parametric:.2f}%（正态假设）')


---

## 2　Bootstrap

### 2.1　Bootstrap 的核心思想

当统计量的解析分布未知（或难以推导）时，
Bootstrap 用**重复抽样**来估计抽样分布：
从原始样本中**有放回**地抽取同等大小的子样本，
重复 $B$ 次，得到 $B$ 个统计量，用其分布近似真实的抽样分布。

**标准 Bootstrap 的局限**：假设样本独立同分布。
对于时序数据（如每日收益率），观测之间存在自相关，
必须用 **Block Bootstrap** 保留局部的时序结构。


In [ ]:
# ── 2.1  夏普比率的 Bootstrap 置信区间 ───────────────────────────────
# 模拟每日收益率（带自相关）
T_ret = 252 * 3   # 3 年
phi   = 0.15      # AR(1) 系数（模拟动量效应）
eps   = RNG.normal(0, 0.01, T_ret)
rets  = np.zeros(T_ret)
for t in range(1, T_ret):
    rets[t] = phi * rets[t-1] + eps[t]
rets += 0.0004    # 正的日均收益

def sharpe(r, ann=252):
    return r.mean() / r.std() * np.sqrt(ann)

sr_obs = sharpe(rets)
print(f'样本夏普比率：{sr_obs:.4f}')

# ── 标准 Bootstrap（忽略自相关，用于对比）─────────────────────────────
B = 2000
sr_boot_iid = []
for _ in range(B):
    idx = RNG.integers(0, T_ret, T_ret)
    sr_boot_iid.append(sharpe(rets[idx]))

ci_iid = np.percentile(sr_boot_iid, [2.5, 97.5])

# ── Block Bootstrap（保留时序结构）─────────────────────────────────────
block_size = 20    # 约 1 个月
sr_boot_blk = []
n_blocks = T_ret // block_size
for _ in range(B):
    blk_idx = RNG.integers(0, n_blocks, n_blocks)
    sample  = np.concatenate([rets[b*block_size:(b+1)*block_size]
                              for b in blk_idx])[:T_ret]
    sr_boot_blk.append(sharpe(sample))

ci_blk = np.percentile(sr_boot_blk, [2.5, 97.5])

print(f'标准 Bootstrap 95% CI ：[{ci_iid[0]:.4f}, {ci_iid[1]:.4f}]')
print(f'Block Bootstrap 95% CI：[{ci_blk[0]:.4f}, {ci_blk[1]:.4f}]')
print(f'（Block Bootstrap 的 CI 更宽，正确反映了时序相关导致的不确定性）')


---

## 3　MCMC 简介

### 3.1　直觉：从难以采样的分布中获取样本

很多贝叶斯后验分布没有解析形式，无法直接采样。
**MCMC（Markov Chain Monte Carlo）** 的思路：
构造一条马尔可夫链，使其稳态分布就是目标分布。
沿链前进足够长之后，链上的点就近似来自目标分布。

**Metropolis-Hastings 算法**（最基础的 MCMC）：

1. 从当前位置 $\theta^{(t)}$ 提议一个新位置 $\theta^*$
2. 计算接受率 $\alpha = \min\left(1, \frac{p(\theta^*)}{p(\theta^{(t)})}\right)$
3. 以概率 $\alpha$ 接受 $\theta^*$，否则停在原地

本课程不要求深入掌握 MCMC，重点在于理解它解决了什么问题，
以及知道 `pymc` 可以用来做贝叶斯推断。


In [ ]:
# ── 3.1  手动 MH 采样演示 ───────────────────────────────────────────────
# 目标：从双峰分布 p(θ) ∝ exp(-θ²/2) + exp(-(θ-4)²/2) 采样

def log_target(theta):
    return np.log(np.exp(-theta**2/2) + np.exp(-(theta-4)**2/2) + 1e-300)

N_chain = 20_000
prop_std = 1.5    # 提议分布标准差
chain = np.zeros(N_chain)
chain[0] = 0.0

n_accept = 0
for t in range(1, N_chain):
    theta_star = chain[t-1] + RNG.normal(0, prop_std)
    log_alpha  = log_target(theta_star) - log_target(chain[t-1])
    if np.log(RNG.uniform()) < log_alpha:
        chain[t] = theta_star
        n_accept += 1
    else:
        chain[t] = chain[t-1]

burnin = N_chain // 4
samples = chain[burnin:]
print(f'接受率：{n_accept/N_chain:.3f}（理想范围：0.2–0.5）')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(chain[:2000], lw=0.5, color='steelblue', alpha=0.7)
ax1.axvline(burnin, color='red', lw=1, linestyle='--', label=f'burn-in={burnin}')
ax1.set_title('MCMC 链的轨迹（前 2000 步）', fontsize=10)
ax1.set_xlabel('迭代次数'); ax1.set_ylabel('θ'); ax1.legend(fontsize=9)

theta_range = np.linspace(-4, 8, 300)
p_true = np.exp(-theta_range**2/2) + np.exp(-(theta_range-4)**2/2)
p_true /= p_true.sum() * (theta_range[1]-theta_range[0])
ax2.hist(samples, bins=60, density=True, color='steelblue',
         edgecolor='white', alpha=0.7, label='MCMC 样本')
ax2.plot(theta_range, p_true, 'r-', lw=2, label='真实目标分布')
ax2.set_title('后验样本 vs 真实分布', fontsize=10)
ax2.set_xlabel('θ'); ax2.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch10_mcmc.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.2  pymc 最小示例：贝叶斯线性回归 ──────────────────────────────────
try:
    import pymc as pm
    import arviz as az
    PYMC_OK = True
    print(f'pymc {pm.__version__} ✓')
except ImportError:
    PYMC_OK = False
    print('pymc 未安装（pip install pymc）')

if PYMC_OK:
    # 生成简单的线性数据
    X_pm = RNG.normal(0, 1, 100)
    Y_pm = 2.5 * X_pm + 1.0 + RNG.normal(0, 0.8, 100)

    with pm.Model() as linear_model:
        alpha = pm.Normal('alpha', mu=0, sigma=10)     # 截距先验
        beta  = pm.Normal('beta',  mu=0, sigma=10)     # 斜率先验
        sigma = pm.HalfNormal('sigma', sigma=2)        # 噪声先验
        mu_pm = alpha + beta * X_pm
        Y_obs = pm.Normal('Y_obs', mu=mu_pm, sigma=sigma, observed=Y_pm)

        trace = pm.sample(1000, tune=500, progressbar=False,
                          return_inferencedata=True, random_seed=42)

    print(az.summary(trace, var_names=['alpha','beta','sigma']).round(4))
    print('真实参数：alpha=1.0  beta=2.5  sigma=0.8')


---

## 4　交叉验证与时序数据

### 4.1　k-fold 交叉验证的时序数据陷阱

标准 k-fold 随机打乱数据划分训练/验证集。
对时序数据，这会让**未来数据出现在训练集**，
模型「看到了」它需要预测的信息，导致**前视偏差（Look-Ahead Bias）**。

验证集指标虚高，模型上线后表现大幅下降——
这是量化策略「回测完美、实盘惨淡」的最常见原因之一。

**正确做法**：用 `TimeSeriesSplit` 或 Walk-Forward Validation，
始终保证训练集在验证集之前。


In [ ]:
# ── 4.1  演示前视偏差：k-fold vs TimeSeriesSplit ─────────────────────
# 生成 AR(1) 时序数据
T_cv  = 500
phi_cv= 0.5
eps_cv= RNG.normal(0, 1, T_cv)
X_cv  = np.zeros(T_cv)
for t in range(1, T_cv):
    X_cv[t] = phi_cv * X_cv[t-1] + eps_cv[t]

# 特征：用过去 5 期的 X 预测当期 X
lag = 5
feat_cv = np.column_stack([X_cv[i:T_cv-lag+i] for i in range(lag)])
target  = X_cv[lag:]

model = Ridge(alpha=1.0)

# k-fold（错误用法）
kf   = KFold(n_splits=5, shuffle=True, random_state=42)
mse_kfold = []
for tr, te in kf.split(feat_cv):
    model.fit(feat_cv[tr], target[tr])
    mse_kfold.append(mean_squared_error(target[te], model.predict(feat_cv[te])))

# TimeSeriesSplit（正确用法）
tscv = TimeSeriesSplit(n_splits=5)
mse_ts = []
for tr, te in tscv.split(feat_cv):
    model.fit(feat_cv[tr], target[tr])
    mse_ts.append(mean_squared_error(target[te], model.predict(feat_cv[te])))

print('前视偏差演示：')
print(f'  k-fold（有前视偏差）MSE 均值：{np.mean(mse_kfold):.4f}  ← 虚假偏低')
print(f'  TimeSeriesSplit MSE 均值   ：{np.mean(mse_ts):.4f}  ← 真实评估')
print(f'  高估比例：{(np.mean(mse_ts)-np.mean(mse_kfold))/np.mean(mse_ts)*100:.1f}%')


In [ ]:
# ── 4.2  可视化：TimeSeriesSplit 的划分方式 ──────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(10, 6), sharex=True)
n = len(feat_cv)
for i, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5).split(feat_cv)):
    ax = axes[i]
    ax.fill_between(range(n), 0, 1,
                    where=np.isin(range(n), tr),
                    color='steelblue', alpha=0.5, label='训练集' if i==0 else '')
    ax.fill_between(range(n), 0, 1,
                    where=np.isin(range(n), te),
                    color='darkorange', alpha=0.7, label='验证集' if i==0 else '')
    ax.set_yticks([])
    ax.set_ylabel(f'Fold {i+1}', fontsize=9, rotation=0, ha='right')

axes[0].legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('时间步', fontsize=10)
plt.suptitle('TimeSeriesSplit：验证集始终在训练集之后', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch10_tscv.png', dpi=150, bbox_inches='tight')
plt.show()


---

## 5　章末练习

**练习 1（MC VaR）**
模拟一个等权 5 资产组合，利用 Cholesky 分解引入相关性，
分别计算 1 天和 10 天的 95%/99% VaR 和 CVaR，
与参数法（方差-协方差法）对比，讨论差异来源。

**练习 2（Block Bootstrap）**
用 Block Bootstrap（block_size = 5 / 10 / 20 天）
计算夏普比率的 95% 置信区间，比较不同 block size 对 CI 宽度的影响，
解释为什么 block size 太小或太大都会带来问题。

**练习 3（前视偏差量化）**
构造一个「完全随机的特征」（和目标变量完全无关），
分别用 k-fold 和 TimeSeriesSplit 评估一个简单线性模型，
对比两种方法的 MSE，直接展示前视偏差「凭空创造」预测能力的程度。

**练习 4（思考题）**
Walk-Forward Validation 和 TimeSeriesSplit 的主要区别是什么？
在什么情况下应该用 Walk-Forward？什么情况下 TimeSeriesSplit 够用？
